# BARD1 analysis

In [ ]:
!pip install -e ~/repos/igvf-cvfg-biobank

File locations are defined here:

In [ ]:
cohorts_dir = 'cohorts'
ancestry_preds_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv'

## Genomic carriers of variant groups

In this section of the notebook, we define the groups of variants of interest; for each group, identify tha participants that carry a variant in the group.

In [ ]:
import pandas as pd
from cvfgaou import hailtools

In [ ]:
sge_df = pd.read_table('20260615_SGEsubset_v10.tsv', dtype=str)
bard1_df = sge_df[
    sge_df['Dataset'] == 'BARD1_IGVF'
][[
    'auth_reported_func_class',
    'simplified_consequence',
    'Chrom',
    'hg38_start',
    'ref_allele',
    'alt_allele'
]]

Establish our variant categories of interest:

In [ ]:
bard1_df[['auth_reported_func_class', 'simplified_consequence']].value_counts().sort_index()

In [ ]:
bard1_df['hg38_start'].isna().any()

In [ ]:
bard1_df['Chromosome'] = 'chr' + bard1_df['Chrom']
bard1_df['pos'] = pd.to_numeric(bard1_df['hg38_start'], downcast='integer')

We will split off as follows:
 - All Functionally Normal
 - Functionally Abnormal
   - All
   - Missense
   - Stop gained
   - Deletion
   - Splicing
   - Stop gained + Deletion + Splicing

In [ ]:
variant_groups = {
    'Functionally Normal': bard1_df[bard1_df['auth_reported_func_class'] == 'Functionally Normal'],
    'Functionally Abnormal - All': bard1_df[bard1_df['auth_reported_func_class'] == 'Functionally Abnormal'],
    'Functionally Abnormal - Missense':
        bard1_df[
            (bard1_df['auth_reported_func_class'] == 'Functionally Abnormal') &
            (bard1_df['simplified_consequence'] == 'missense_variant')
        ],
    'Functionally Abnormal - Stop Gained':
        bard1_df[
            (bard1_df['auth_reported_func_class'] == 'Functionally Abnormal') &
            (bard1_df['simplified_consequence'] == 'stop_gained')
        ],
    'Functionally Abnormal - Deletion':
        bard1_df[
            (bard1_df['auth_reported_func_class'] == 'Functionally Abnormal') &
            (bard1_df['simplified_consequence'] == 'inframe_deletion')
        ],
    'Functionally Abnormal - Splicing':
        bard1_df[
            (bard1_df['auth_reported_func_class'] == 'Functionally Abnormal') &
            (bard1_df['simplified_consequence'].str.startswith('splice_'))
        ],
    'Functionally Abnormal - Stop Gained / Deletion / Splicing':
        bard1_df[
            (bard1_df['auth_reported_func_class'] == 'Functionally Abnormal') & (
                bard1_df['simplified_consequence'].isin((
                    'stop_gained', 'inframe_deletion'
                )) |
                bard1_df['simplified_consequence'].str.startswith('splice_')
            )
        ]
}

In [ ]:
import os
import hail as hl
from tqdm.notebook import tqdm

In [ ]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=os.getenv('GOOGLE_PROJECT')
)

In [ ]:
# WGS table

wgs_mt = hl.read_matrix_table(
    "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"
)

In [ ]:
carrier_dfs = {
    variant_group:
        hailtools.get_cols_with_variants(
            df, wgs_mt,
            contig_col='Chromosome',
            pos_col='pos',
            ref_col='ref_allele',
            alt_col='alt_allele',
            reference_genome='GRCh38'
        )
    for variant_group, df in tqdm(variant_groups.items())
}

In [ ]:
pd.concat(carrier_dfs, names=['Variant group']).to_parquet('func-analysis-BARD1/carriers.parquet')

## Phenotypic cohort

In this section we identify cases and controls for BARD1-related phenotypes.

In [ ]:
# We may repeat some imports here in case we want to run only a section of the notebook
import os
import pandas as pd

In [ ]:
from pandas_gbq import read_gbq

def load_or_query_and_save(file_path, sql, overwrite=False):
    """Helper function for loading queries.
    We load a table from a query and save it to a file if the file doesn't exist.
    If the file exists and the overwrite flag isn't True, we load from the file instead.
    """
    
    df = None
    if not overwrite:
        try:
            # Try to load saved table
            df = pd.read_table(file_path)
            print('Loaded from file.')

        except FileNotFoundError:
            print('File not found.')
    
    if df is None:
        print('Querying DB...')
        df = read_gbq(
            sql,
            dialect="standard",
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            progress_bar_type="tqdm_notebook")
        df.to_csv(file_path, sep='\t', index=False)
        print('...saved to file.')
    
    return df

In [ ]:
os.environ["WORKSPACE_CDR"]

### Case Cohort

#### Breast cancer

In [ ]:
# This query represents dataset "Breast cancer case pool" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_51645958_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (4112853)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (836772) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528500))) criteria ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
            WHERE
                sex_at_birth_concept_id IN (45878463) ) )"""


# This query represents dataset "Breast cancer case pool" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_51645958_survey_sql = """
    SELECT
        answer.person_id,
        answer.answer_concept_id,
        answer.answer  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (43530444)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (4112853)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria 
                UNION
                DISTINCT SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN (836772) 
                        AND is_standard = 0  
                        AND  value_source_concept_id IN (43528500))) criteria ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
                WHERE
                    sex_at_birth_concept_id IN (45878463) ) )
            )"""


# This query represents dataset "Breast cancer case pool" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_51645958_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4112853)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4112853)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN (836772) 
                            AND is_standard = 0  
                            AND  value_source_concept_id IN (43528500))) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
                    WHERE
                        sex_at_birth_concept_id IN (45878463) ) )
                )
            ) c_occurrence"""

In [ ]:
load_or_query_and_save(
    f'{cohorts_dir}/breast cancer case pool.tsv.gz',
    dataset_51645958_person_sql,
)

#### Ovarian cancer

In [ ]:
# This query represents dataset "Ovarian cancer case pool" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_12955955_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (4181351)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (836778) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1384488))) criteria ) )"""


In [ ]:
# This query represents dataset "Ovarian cancer case pool" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_12955955_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4181351)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4181351)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN (836778) 
                            AND is_standard = 0  
                            AND  value_source_concept_id IN (1384488))) criteria ) )
                )
            ) c_occurrence"""


In [ ]:
# This query represents dataset "Ovarian cancer case pool" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_12955955_survey_sql = """
    SELECT
        answer.person_id,
        answer.answer_concept_id,
        answer.answer  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1740639)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (4181351)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria 
                UNION
                DISTINCT SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN (836778) 
                        AND is_standard = 0  
                        AND  value_source_concept_id IN (1384488))) criteria ) )
            )"""


In [ ]:
load_or_query_and_save(
    f'{cohorts_dir}/ovarian cancer case pool.tsv.gz',
    dataset_12955955_person_sql,
)

#### Control cohort

In [ ]:
# This query represents dataset "Cancer control pool" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_21635573_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (443392)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (836768) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528472) 
                    OR  concept_id IN (836769) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1384404) 
                    OR  concept_id IN (836770) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528480) 
                    OR  concept_id IN (836771) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528492) 
                    OR  concept_id IN (836772) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528500) 
                    OR  concept_id IN (836773) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528539) 
                    OR  concept_id IN (836831) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528565) 
                    OR  concept_id IN (836832) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528668) 
                    OR  concept_id IN (836833) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528679) 
                    OR  concept_id IN (836774) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528693) 
                    OR  concept_id IN (836775) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528695) 
                    OR  concept_id IN (836834) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43528886) 
                    OR  concept_id IN (836776) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529136) 
                    OR  concept_id IN (836777) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529184) 
                    OR  concept_id IN (836778) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1384488) 
                    OR  concept_id IN (836779) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529684) 
                    OR  concept_id IN (836780) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529733) 
                    OR  concept_id IN (836781) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529817) 
                    OR  concept_id IN (836782) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529866) 
                    OR  concept_id IN (836783) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529909) 
                    OR  concept_id IN (836835) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (43529624))) criteria ) )"""

cancer_control_pool_df = load_or_query_and_save(
    f'{cohorts_dir}/cancer control pool.tsv.gz',
    dataset_21635573_person_sql)

### Demographic covariates and ancestry

#### Demographics

In [ ]:
# Get event count statistics

count_events_sql = f"""
    SELECT *
    FROM (
        SELECT
            person_id,
            COUNT(DISTINCT condition_start_date) AS n_events,
            MIN(condition_start_date) AS first_event,
            MAX(condition_start_date) AS last_event
        FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence`
        WHERE
            person_id IN (
                SELECT person_id FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_person`
                WHERE has_whole_genome_variant = 1
            )
        GROUP BY person_id
    ) INNER JOIN (
        SELECT
            person_id,
            sex_at_birth,
            dob,
            age_at_cdr
        FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_person`
    ) USING (person_id)
"""

count_events_df = load_or_query_and_save(
    f'{cohorts_dir}/demo.tsv.gz',
    count_events_sql
)

#### Ancestry PCA table

In [ ]:
ancestry_df = pd.read_table(
    ancestry_preds_path,
    storage_options={'requester_pays': True}
)

In [ ]:
ancestry_df

In [ ]:
# Ancestry PCA
from ast import literal_eval

pca_df = pd.concat(
    [
        ancestry_df[['research_id']],
        ancestry_df.pca_features.apply(lambda x: pd.Series(literal_eval(x))).rename(columns=lambda c: f'pca_{c}')
    ],
    axis='columns'
)

In [ ]:
pca_df

In [ ]:
pca_df.to_csv(f'{cohorts_dir}/ancestry_pca.tsv.gz', sep='\t', index=False)

## Odds Ratio estimation

In [ ]:
import pandas as pd
from cvfgaou import aou, odds

In [ ]:
# Gene case pools definitions
bard1_phenotype = {
    'BARD1': (
        {'breast cancer', 'ovarian cancer'},
        {'cancer'},
        'autosomal dominant'
    )
}

cohorts = aou.CohortLoader(
    gene_cohort_map = {
        gene: (
            {f'{p} case pool.tsv.gz' for p in case_phenos},
            {f'{p} control pool.tsv.gz' for p in control_phenos}
        )
        for gene, (case_phenos, control_phenos, _) in bard1_phenotype.items()
    },
    ancestry_df = pd.read_table(f'{cohorts_dir}/ancestry_pca.tsv.gz', index_col='research_id'),
    demo_df = pd.read_table(f'{cohorts_dir}/demo.tsv.gz', index_col='person_id'),
    data_dir = cohorts_dir
)
    

In [ ]:
# Load the combined carrier DF we created
carrier_df = pd.read_parquet('func-analysis-BARD1/carriers.parquet')

In [ ]:
or_estimates_dict = {
    variant_group:
        odds.estimate_logOR(
            pd.to_numeric(exposure_df.s, downcast='integer'),
            cohorts.gene_cohort('BARD1')
        )
    for variant_group, exposure_df in carrier_df.groupby('Variant group')
}

In [ ]:
or_estimates_df = pd.DataFrame.from_dict(or_estimates_dict, orient='index')

In [ ]:
or_estimates_df

In [ ]:
or_estimates_df.to_parquet('func-analysis-BARD1/raw-or-estimates.parquet')

Regroup count variants as totals + carrier counts for reporting

In [ ]:
or_df = or_estimates_df.copy()
or_df['Total cases'] = or_df['Cases with variants'] + or_df['Cases without variants']
or_df['Total controls'] = or_df['Controls with variants'] + or_df['Controls without variants']

In [ ]:
or_df['Carrier cases'] = or_df['Cases with variants'].astype(str).where(
    (or_df['Cases with variants'] <= 20) & (or_df['Cases with variants'] > 0),
    '≤ 20'
)

In [ ]:
or_df['Carrier controls'] = or_df['Controls with variants'].astype(str).mask(
    (or_df['Controls with variants'] <= 20) & (or_df['Controls with variants'] > 0),
    '≤ 20'
)

In [ ]:
deliverable_df = or_df[
    [
        'LogOR',
        'LogOR_LI',
        'LogOR_UI',
        'p-value',
        'Cohort size',
        'Total cases',
        'Total controls',
        'Carrier cases',
        'Carrier controls'
    ]
].rename(columns={
    'p-value': 'Wald p-value'
})

deliverable_df

In [ ]:
deliverable_df.to_csv('func-analysis-BARD1/BARD1_functional-classes_odds-ratio-estimates.csv')